In [ ]:
from transformers import DetrForObjectDetection, DetrImageProcessor
from PIL import Image
import torch

In [ ]:
image_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")

In [ ]:
import requests
from PIL import Image
from io import BytesIO
import torch
from torchvision import transforms

# Load image from URL
url ="http://images.cocodataset.org/val2017/000000039769.jpg"
#url = "https://www.cogito-ntnu.no/_next/image?url=%2F_next%2Fstatic%2Fmedia%2FSverreOgOlav.1c79ee15.jpg&w=3840&q=75"
#url = "https://veggipedia-cms.production.taks.zooma.cloud/assets/Uploads/Products/Broccoli-groenten-veggipedia.png"
response = requests.get(url)
img = Image.open(BytesIO(response.content)).convert("RGB")



In [ ]:
inputs = image_processor(images=img, return_tensors="pt")
outputs = model(**inputs)
results = image_processor.post_process_object_detection(outputs, threshold=0.9, target_sizes=[(img.height, img.width)])

In [ ]:
from matplotlib import pyplot as plt
# COCO classes
CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]

# colors for visualization
COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]
          
def plot_results(pil_img, prob, labels, boxes):
    plt.figure(figsize=(16,10))
    plt.imshow(pil_img)
    ax = plt.gca()
    print(labels.tolist())
    for p, label, (xmin, ymin, xmax, ymax), c in zip(prob, labels.tolist(), boxes.tolist(), COLORS * 100):
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=3))
        cl = p.argmax()
        text = f'{CLASSES[int(label)]}: {p:0.2f}'
        ax.text(xmin, ymin, text, fontsize=15,
                bbox=dict(facecolor='yellow', alpha=0.5))
    plt.axis('off')
    plt.show()

In [ ]:
plot_results(img, results[0]["scores"], results[0]["labels"], results[0]["boxes"])